## Process
- Load the CNOUS export
- Clean data and map it to the PSP schema
- Filter on the boursier birthdate window and drop duplicates
- Serialize the allocataire and address JSON columns
- Add the production default columns, draw one pass Sport code per beneficiary
- Output to CSV

CNOUS is the one partner where the beneficiary **is** the allocataire: a boursier applies
for themselves. There is no household to resolve, so no `quotient_familial` call and no
qf-batch checkpoint - a single notebook takes the raw export all the way to the file that
gets injected.

For the same reason this notebook draws its own codes rather than going through
`generate_new_codes.ipynb` (step ③): it is already the last step. It seeds the drawing with
`EXISTING_CODES_PATHFILE_2026` and rewrites it, exactly like step ③ does, so codes stay
unique across every run of the campaign.

## Encoding
CNOUS -> utf-8, sep=`;`

In [ ]:
import csv
import os
import sys
from pathlib import Path

import pandas as pd
from dotenv import load_dotenv

# partners_lib imports utils.data_utils, which lives at the data/ root: make that root
# importable first, since this notebook runs from its own directory.
try:
    import utils.data_utils  # noqa: F401
except ModuleNotFoundError:
    for parent in [Path.cwd(), *Path.cwd().parents]:
        if (parent / "utils" / "data_utils.py").exists():
            sys.path.append(str(parent))
            break

# partners_lib and generate_codes_lib sit one level up, in partners/, next to the other
# partner folders.
partners_root = str(Path.cwd().parent)
if partners_root not in sys.path:
    sys.path.append(partners_root)

# All the DataFrame processing lives in two modules so it can be unit tested: what every
# partner shares in ../partners_lib.py, and what is specific to the CNOUS file (its raw
# columns, its two date formats, its ISO birth country) in clean_cnous_lib.py. The default
# columns and the code drawing are the very same ones step ③ runs for the other partners.
import clean_cnous_lib as cnous
import generate_codes_lib as codes
import partners_lib as partners
from utils.codes_utils import load_existing_codes, save_codes

load_dotenv()

cnous_input_filepath = os.environ['CNOUS_PATHFILE_2026']
base_output_filepath = os.environ['DB_CNOUS_EXPORT_2026']

# Single-column ("code") CSV holding every code already handed out this campaign. Read
# before drawing anything and rewritten at the end - that is what keeps the successive runs
# of this notebook and of generate_new_codes.ipynb from colliding.
existing_codes_filepath = os.environ['EXISTING_CODES_PATHFILE_2026']

In [ ]:
# CNOUS - column names are supplied positionally, not read from the file's own header row
# (see clean_cnous_lib.read_raw_cnous_csv: that row names 19 columns for 20 delivered
# fields, the trailing bourse échelon being unnamed).
df_cnous = cnous.read_raw_cnous_csv(cnous_input_filepath)

print(f"{len(df_cnous)} row(s) read from {cnous_input_filepath}")

In [ ]:
# clean white spaces within all columns, map to the PSP schema (see
# cnous.CNOUS_COLUMN_MAPPING), then flag the organism and the situation - constant here,
# every row of the file is a boursier.
df_psp_mapped = partners.strip_all_string_columns(df_cnous)
df_psp_mapped = cnous.map_cnous_columns(df_psp_mapped)
df_psp_mapped = cnous.set_organisme_and_situation(df_psp_mapped)

# Allocataire's qualite: MME/MR -> Mme/M, the shape CNAF and MSA also write
df_psp_mapped = partners.normalize_allocataire_qualite(df_psp_mapped)

In [ ]:
# Beneficiary birthdate to a datetime python object for processing - CNOUS delivers it
# ISO, where CNAF and MSA use %d/%m/%Y. The allocataire's own stays text, in the format the
# allocataire JSON has carried since 2025.
df_psp_mapped = partners.parse_beneficiary_birthdate(df_psp_mapped, date_format='%Y-%m-%d')
df_psp_mapped = cnous.normalize_allocataire_birthdate(df_psp_mapped)

unparsed_dob = df_psp_mapped['allocataire-date_naissance'].isna().sum()
print(f"{unparsed_dob} allocataire birthdate(s) unparsable, dropped from the allocataire JSON")

In [ ]:
# Birth country: an empty or legacy '100' ISO code means France, and the label is read off
# the code. An ISO code absent from the reference table leaves the label empty rather than
# killing the run.
df_psp_mapped = cnous.fill_default_birth_country_iso(df_psp_mapped)
df_psp_mapped = cnous.add_birth_country_label(df_psp_mapped)
df_psp_mapped = cnous.normalize_birthplace_casing(df_psp_mapped)

unmapped_iso = sorted(set(df_psp_mapped.loc[
    df_psp_mapped['allocataire-pays_naissance'].isna(),
    'allocataire-code_iso_pays_naissance'].dropna()))
if unmapped_iso:
    print(f"{len(unmapped_iso)} ISO birth-country code(s) without a country label: {unmapped_iso}")

In [ ]:
# Restore the leading zero the export drops from the INSEE and postal codes of the
# départements 01-09 (Péronnas: commune 1289 -> 01289). Done on the frame because the
# allocataire JSON takes its extra fields verbatim.
df_psp_mapped = cnous.pad_insee_codes(df_psp_mapped)

In [ ]:
# remove rows with missing necessary values (if one of those value are missing we cannot
# generate a code), then columns with all null value
df_valid = partners.filter_rows_missing_required_fields(df_psp_mapped)

print(f"{len(df_psp_mapped) - len(df_valid)} row(s) removed for a missing "
      f"{partners.NECESSARY_COLUMNS}")

In [ ]:
# Upper case the identity columns for the merge, lower case the emails, and unaccent /
# upper case the address text the way CNOUS stores it.
df_valid = partners.normalize_identity_casing(df_valid)
df_valid = partners.normalize_email_casing(df_valid)
df_valid = cnous.normalize_address_casing(df_valid)

In [ ]:
# Boursier birthdate window for the campaign (cnous.CNOUS_DOB_MIN / CNOUS_DOB_MAX),
# bounds included.
df_valid_after, out_of_window_count = cnous.filter_within_birthdate_window(df_valid)

print(f"{out_of_window_count} row(s) removed because they are outside the birthdate window "
      f"{cnous.CNOUS_DOB_MIN:%d/%m/%Y} - {cnous.CNOUS_DOB_MAX:%d/%m/%Y}")
cnous.describe_rows_outside_window(df_valid)

In [ ]:
# set NaN values for not existing courriel, then add 4h on all birthdates so a timezone
# conversion down the line cannot roll them onto the previous day
df_valid_after = partners.clear_blank_email(df_valid_after)
df_valid_after = partners.shift_birthdate_by_hours(df_valid_after)

In [ ]:
# Deduplicate on the INE first, then on the courriel. Rows without a value for the key are
# left alone: two boursiers the export has no email for are two beneficiaries, not one (see
# cnous.drop_duplicates_on_key).
df_valid_no_duplicate, duplicate_ine_count = cnous.drop_duplicates_on_key(
    df_valid_after, 'allocataire-matricule')
print(f"{duplicate_ine_count} row(s) sharing an INE were removed")

df_valid_no_duplicate, duplicate_email_count = cnous.drop_duplicates_on_key(
    df_valid_no_duplicate, 'allocataire-courriel')
print(f"{duplicate_email_count} row(s) sharing a courriel were removed")

print(f"{len(df_valid_no_duplicate)} beneficiary row(s) left")

In [ ]:
# map allocataire json (+ the boursier's own birth details, cnous.ALLOCATAIRE_JSON_EXTRA_FIELDS)
# and adresse_allocataire json. The code organisme and telephone columns the shared
# serializer indexes are added here, empty: CNOUS delivers neither, and null values are
# dropped from the JSON. Added now rather than earlier, so the all-null column drop above
# cannot take them away again.
df_final = cnous.add_missing_allocataire_columns(df_valid_no_duplicate)
df_final = partners.add_allocataire_json_column(
    df_final, extra_fields=cnous.ALLOCATAIRE_JSON_EXTRA_FIELDS)
df_final = partners.add_adresse_allocataire_json_column(df_final)

In [ ]:
# The columns the production table expects but no cleaning step produces: exercice_id,
# uuid_doc, the zrr/qpv/a_valider/refuser flags at False and the timestamps. Same function
# step ③ runs for the other partners, so the campaign's rows cannot drift apart.
df_final = codes.add_production_default_columns(df_final)

# One brand new code per beneficiary, drawn against every code already handed out this
# campaign. The code list itself is only rewritten once the output file is safely on disk.
existing_codes = load_existing_codes(existing_codes_filepath)
df_final = codes.assign_new_codes(df_final, existing_codes)

print(f"{len(existing_codes)} existing code(s) read from {existing_codes_filepath}")

In [ ]:
# Cast to string
df_final.loc[:, 'date_naissance'] = df_final['date_naissance'].astype(str)

# output to CSV file - the 8 first columns are exactly clean_cnaf_2's df_final_jeune, the
# rest is what CNOUS adds itself in place of step ③ (see cnous.CNOUS_OUTPUT_COLUMNS)
df_final_jeune = cnous.select_output_columns(df_final)
df_final_jeune.to_csv(
    base_output_filepath, sep=';', index=False, encoding='utf-8', quoting=csv.QUOTE_ALL)

print(f"{len(df_final_jeune)} df_final_jeune written to {base_output_filepath}")

In [ ]:
# Track the codes just handed out, now that the output file is written: a run that died
# midway must not burn codes it never gave anyone.
tracked_codes = save_codes(existing_codes_filepath, existing_codes | set(df_final_jeune['id_psp']))
assert tracked_codes == len(existing_codes) + len(df_final_jeune)

print(f"{tracked_codes} code(s) now tracked in {existing_codes_filepath}")

In [ ]:
# Run report, the same counts step ③ prints for the other partners
print(f"beneficiaires: {len(df_final_jeune)}")
print(f"codes_generes: {len(df_final_jeune)}")
print(f"genre_M: {len(df_final_jeune[df_final_jeune['genre'] == 'M'])}")
print(f"genre_F: {len(df_final_jeune[df_final_jeune['genre'] == 'F'])}")
print(df_final_jeune[['organisme', 'situation']].value_counts())